# Experiment: Case-001 July 10 active registry gate

Objective: classify fresh ClinicalTrials.gov and PubMed source snapshots into public source-radar labels only, with no eligibility, referral, trial-screening, or treatment outputs.

In [ ]:
import json
from collections import Counter
from pathlib import Path

repo = Path.cwd()
registry_dir = repo / "data/registries/clinicaltrials"
pubmed_dir = repo / "data/literature/pubmed"

china_file = "2026-07-10-china-active-beta-thalassemia-refresh.json"
cs101_file = "2026-07-10-cs101-clinicaltrials-refresh.json"
asia_null_file = "2026-07-10-asia-location-query-null.json"
pubmed_file = "2026-07-10-asia-thalassemia-current-therapy-refresh.json"

china = json.loads((registry_dir / china_file).read_text(encoding="utf-8"))
cs101 = json.loads((registry_dir / cs101_file).read_text(encoding="utf-8"))
asia_null = json.loads((registry_dir / asia_null_file).read_text(encoding="utf-8"))
pubmed = json.loads((pubmed_dir / pubmed_file).read_text(encoding="utf-8"))

assert china["total_count"] == 23
assert cs101["total_count"] == 8
assert asia_null["total_count"] == 0
assert pubmed["count"] == 10
(china["total_count"], cs101["total_count"], asia_null["total_count"])

## Plan

- Use the China active query as the positive source-radar set.
- Use the CS-101 query as a focused editing benchmark set.
- Treat `query.locn=Asia` returning zero as a query-quality warning, not evidence of no Asia records.
- Emit only source labels and owner questions.

In [ ]:
def text_blob(record: dict) -> str:
    """Combine searchable registry fields for lane classification."""
    parts = [record.get("title") or ""]
    parts.extend(record.get("interventions") or [])
    return " ".join(parts).lower()


def lane_label(record: dict) -> str:
    """Assign a source-radar lane without patient-action meaning."""
    text = text_blob(record)
    if "cs-101" in text:
        return "cs101_base_editing_benchmark"
    if any(term in text for term in ("bd211", "brl-101", "kl003", "lentired")):
        return "autologous_gene_cell_benchmark"
    if any(term in text for term in ("yolt", "gene modified", "gene therapy")):
        return "gene_cell_benchmark"
    if any(term in text for term in ("supergraft", "transplant")):
        return "transplant_strategy_benchmark"
    if any(term in text for term in ("luspatercept", "ace-536")):
        return "luspatercept_benchmark"
    return "other_active_source"


status_counts = Counter(record["status"] for record in china["records"])
lane_counts = Counter(lane_label(record) for record in china["records"])

assert status_counts == {
    "RECRUITING": 16,
    "NOT_YET_RECRUITING": 3,
    "ENROLLING_BY_INVITATION": 4,
}
dict(lane_counts)

In [ ]:
cs101_records = {record["nct_id"]: record for record in cs101["records"]}

required_ids = {
    "NCT06291961",
    "NCT07489196",
    "NCT06328764",
    "NCT06479616",
}
assert required_ids <= set(cs101_records)
assert cs101_records["NCT06291961"]["status"] == "COMPLETED"
assert cs101_records["NCT07489196"]["status"] == "NOT_YET_RECRUITING"
assert cs101_records["NCT06328764"]["last_update"] == "2026-06-09"
assert cs101_records["NCT06479616"]["status"] == "RECRUITING"

cs101_source_states = {
    nct_id: {
        "status": cs101_records[nct_id]["status"],
        "last_update": cs101_records[nct_id]["last_update"],
    }
    for nct_id in sorted(required_ids)
}
cs101_source_states

In [ ]:
allowed_outputs = {
    "active_registry_source_radar_ready_benchmark_only",
    "country_specific_query_required",
    "owner_verification_required",
    "branch_review_handoff_packet_incomplete",
}
blocked_outputs = {
    "diagnosis",
    "eligibility",
    "trial_screening",
    "referral",
    "travel",
    "import",
    "purchase",
    "dosing",
    "treatment_selection",
    "sample_routing",
}
assert not (allowed_outputs & blocked_outputs)

summary = {
    "label": "active_registry_source_radar_ready_benchmark_only",
    "china_active_total": china["total_count"],
    "cs101_total": cs101["total_count"],
    "asia_location_total": asia_null["total_count"],
    "query_warning": "do_not_treat_query.locn=Asia_zero_as_absence",
    "pubmed_results": pubmed["count"],
    "case001_state": "branch_review_handoff_packet_incomplete",
}
summary

## Result

- China country-specific ClinicalTrials.gov search is the useful active source-radar query.
- The broad `Asia` location query is a false-negative risk and should not be used as absence evidence.
- CS-101 remains a fast-moving benchmark lane with completed, invitation-only, recruiting, and not-yet-recruiting records.
- The only allowed action is source disambiguation and qualified owner verification.